In [4]:
import pandas as pd
import numpy as np
import random
from pathlib import Path


# Seed data: Verified base rows

seed_rows = [
    {"property_name": "Terrapin Row",       "bedrooms": 2, "bathrooms": 2, "sqft": 982,  "price_per_bed_usd": 959,  "distance_to_campus_miles": 0.3, "walk_time_minutes": 6,  "lease_type": "by-bedroom", "availability": "Aug 2026 Lease", "year": 2026},
    {"property_name": "Terrapin Row",       "bedrooms": 4, "bathrooms": 4, "sqft": 1369, "price_per_bed_usd": 1279, "distance_to_campus_miles": 0.3, "walk_time_minutes": 6,  "lease_type": "by-bedroom", "availability": "Aug 2026 Lease", "year": 2026},
    {"property_name": "Terrapin Row",       "bedrooms": 4, "bathrooms": 2, "sqft": 1118, "price_per_bed_usd": 1099, "distance_to_campus_miles": 0.3, "walk_time_minutes": 6,  "lease_type": "by-bedroom", "availability": "Aug 2026 Lease", "year": 2026},

    {"property_name": "Tempo College Park", "bedrooms": 1, "bathrooms": 1, "sqft": 971,  "price_per_bed_usd": 1137, "distance_to_campus_miles": 0.4, "walk_time_minutes": 8,  "lease_type": "by-bedroom", "availability": "Available",       "year": 2026},
    {"property_name": "Tempo College Park", "bedrooms": 4, "bathrooms": 4, "sqft": 1400, "price_per_bed_usd": 1200, "distance_to_campus_miles": 0.4, "walk_time_minutes": 8,  "lease_type": "by-bedroom", "availability": "Aug 2026 Lease", "year": 2026},
    {"property_name": "Tempo College Park", "bedrooms": 5, "bathrooms": 5, "sqft": 1493, "price_per_bed_usd": 1250, "distance_to_campus_miles": 0.4, "walk_time_minutes": 8,  "lease_type": "by-bedroom", "availability": "Aug 2026 Lease", "year": 2026},

    {"property_name": "University View",    "bedrooms": 4, "bathrooms": 2, "sqft": 1180, "price_per_bed_usd": 1129, "distance_to_campus_miles": 0.6, "walk_time_minutes": 12, "lease_type": "by-bedroom", "availability": "Available",       "year": 2026},
    {"property_name": "University View",    "bedrooms": 2, "bathrooms": 2, "sqft": 960,  "price_per_bed_usd": 1200, "distance_to_campus_miles": 0.6, "walk_time_minutes": 12, "lease_type": "by-bedroom", "availability": "Available",       "year": 2026},

    {"property_name": "The Varsity",        "bedrooms": 4, "bathrooms": 4, "sqft": 1300, "price_per_bed_usd": 1150, "distance_to_campus_miles": 0.5, "walk_time_minutes": 10, "lease_type": "by-bedroom", "availability": "Aug 2026 Lease", "year": 2026},
    {"property_name": "The Varsity",        "bedrooms": 3, "bathrooms": 3, "sqft": 1112, "price_per_bed_usd": 1180, "distance_to_campus_miles": 0.5, "walk_time_minutes": 10, "lease_type": "by-bedroom", "availability": "Aug 2026 Lease", "year": 2026},

    {"property_name": "Landmark College Park", "bedrooms": 4, "bathrooms": 4, "sqft": 1200, "price_per_bed_usd": 1099, "distance_to_campus_miles": 0.4, "walk_time_minutes": 9,  "lease_type": "by-bedroom", "availability": "Aug 2026 Lease", "year": 2026},
    {"property_name": "The Nine",              "bedrooms": 4, "bathrooms": 4, "sqft": 1250, "price_per_bed_usd": 1050, "distance_to_campus_miles": 0.4, "walk_time_minutes": 9,  "lease_type": "by-bedroom", "availability": "Aug 2026 Lease", "year": 2026},
    {"property_name": "University Club",       "bedrooms": 4, "bathrooms": 4, "sqft": 1200, "price_per_bed_usd": 900,  "distance_to_campus_miles": 1.0, "walk_time_minutes": 18, "lease_type": "by-bedroom", "availability": "Aug 2026 Lease", "year": 2026},
    {"property_name": "Vie Towers",            "bedrooms": 4, "bathrooms": 2, "sqft": 1100, "price_per_bed_usd": 850,  "distance_to_campus_miles": 1.6, "walk_time_minutes": 30, "lease_type": "by-bedroom", "availability": "Available",       "year": 2026},
    {"property_name": "Mazza GrandMarc",       "bedrooms": 4, "bathrooms": 4, "sqft": 1250, "price_per_bed_usd": 1050, "distance_to_campus_miles": 0.5, "walk_time_minutes": 10, "lease_type": "by-bedroom", "availability": "Aug 2026 Lease", "year": 2026},
    {"property_name": "College Park Towers",   "bedrooms": 2, "bathrooms": 1, "sqft": 900,  "price_per_bed_usd": 950,  "distance_to_campus_miles": 0.7, "walk_time_minutes": 14, "lease_type": "whole-unit",  "availability": "Available",       "year": 2026},
]


In [5]:
seed_df = pd.DataFrame(seed_rows)


# Config

N_ROWS = 1000
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Optional property-level price adjustment to create richer variation
property_price_bias = {
    "Terrapin Row": 1.05,
    "Tempo College Park": 1.04,
    "University View": 1.02,
    "The Varsity": 1.03,
    "Landmark College Park": 1.01,
    "The Nine": 1.00,
    "University Club": 0.94,
    "Vie Towers": 0.90,
    "Mazza GrandMarc": 0.99,
    "College Park Towers": 0.95,
}

availability_options = [
    "Available",
    "Aug 2026 Lease",
    "Immediate Move-In",
    "Fall 2026",
    "Waitlist",
]

availability_weights = [0.28, 0.42, 0.10, 0.15, 0.05]

year_options = [2025, 2026, 2027]
year_weights = [0.15, 0.65, 0.20]



In [6]:
# Helper functions

def clamp(value, low, high):
    return max(low, min(high, value))

def perturb_numeric(base, pct_std=0.05, min_change=0):
    """
    Multiplicative Gaussian perturbation.
    """
    val = base * (1 + np.random.normal(0, pct_std))
    if min_change:
        if abs(val - base) < min_change:
            val = base + np.sign(np.random.randn()) * min_change
    return val

def generate_walk_time(distance_miles):
    """
    Rough mapping from miles to walking minutes with noise.
    Around 20 min/mile is a decent estimate near campus.
    """
    walk = distance_miles * 20 + np.random.normal(0, 2.0)
    return int(round(clamp(walk, 4, 40)))



In [7]:
def generate_row_from_seed(seed_row, new_id):
    row = seed_row.copy()

    # Slightly varying bedrooms/bathrooms only sometimes, and only in realistic ways
    bedrooms = int(row["bedrooms"])
    bathrooms = float(row["bathrooms"])

    # Most rows Im keeping the original layout

    if random.random() < 0.15:
        if bedrooms > 1 and random.random() < 0.5:
            bedrooms = bedrooms
        else:
            bedrooms = bedrooms

    # Bathrooms generally track bedrooms in student housing
    if row["lease_type"] == "by-bedroom":
        if bedrooms >= 4:
            bathrooms = random.choice([2, 4, 4, 4, 3.5])
        elif bedrooms == 3:
            bathrooms = random.choice([2, 3, 3])
        elif bedrooms == 2:
            bathrooms = random.choice([1, 2, 2])
        else:
            bathrooms = 1
    else:
        bathrooms = random.choice([1, 1, 1.5, 2])

    # Sqft grows with size, but centered on seed row
    sqft = perturb_numeric(row["sqft"], pct_std=0.06, min_change=10)
    sqft = int(round(clamp(sqft, 300, 2200)))

    # Price per bed: seed-based + property bias + year effect + noise
    base_price = row["price_per_bed_usd"]
    bias = property_price_bias.get(row["property_name"], 1.0)
    year = random.choices(year_options, weights=year_weights, k=1)[0]
    year_multiplier = {2025: 0.97, 2026: 1.00, 2027: 1.04}[year]

    price = base_price * bias * year_multiplier * (1 + np.random.normal(0, 0.07))
    price = int(round(clamp(price, 650, 1800)))

    # Distance: keep almost fixed per property with tiny noise
    dist = row["distance_to_campus_miles"] + np.random.normal(0, 0.04)
    dist = round(clamp(dist, 0.2, 2.0), 2)

    walk_time = generate_walk_time(dist)

    availability = random.choices(
        availability_options, weights=availability_weights, k=1
    )[0]

    # Keep lease type mostly consistent
    lease_type = row["lease_type"]

    return {
        "id": new_id,
        "property_name": row["property_name"],
        "bedrooms": bedrooms,
        "bathrooms": bathrooms,
        "sqft": sqft,
        "price_per_bed_usd": price,
        "distance_to_campus_miles": dist,
        "walk_time_minutes": walk_time,
        "lease_type": lease_type,
        "availability": availability,
        "year": year,
    }

In [8]:
# Generate synthetic dataset

generated_rows = []

# Sample seed rows with replacement so bigger properties can recur
sampling_weights = np.array([
    1.3, 1.3, 1.2,   # Terrapin Row
    1.3, 1.2, 1.1,   # Tempo
    1.1, 1.0,        # UView
    1.1, 1.0,        # Varsity
    0.9,             # Landmark
    0.9,             # The Nine
    0.8,             # University Club
    0.7,             # Vie Towers
    0.8,             # Mazza
    0.7,             # CPT
], dtype=float)
sampling_weights = sampling_weights / sampling_weights.sum()

seed_indices = np.random.choice(seed_df.index, size=N_ROWS, replace=True, p=sampling_weights)

for i, idx in enumerate(seed_indices, start=1):
    seed_row = seed_df.loc[idx].to_dict()
    generated_rows.append(generate_row_from_seed(seed_row, i))

df = pd.DataFrame(generated_rows)

In [9]:
# Optional dedup / clean

# Keep rows realistic
df["bathrooms"] = df["bathrooms"].apply(lambda x: int(x) if float(x).is_integer() else x)

# Sort for nicer output
df = df.sort_values(
    by=["property_name", "bedrooms", "bathrooms", "price_per_bed_usd"]
).reset_index(drop=True)

# Reassign clean IDs after sorting
df["id"] = np.arange(1, len(df) + 1)

#Save

output_path = Path("umd_housing_1000rows_synthetic.csv")
df.to_csv(output_path, index=False)

print(f"Saved {len(df)} rows to {output_path.resolve()}")
print(df.head(10))

Saved 1000 rows to /content/umd_housing_1000rows_synthetic.csv
   id        property_name  bedrooms  bathrooms  sqft  price_per_bed_usd  \
0   1  College Park Towers         2        1.0   869                739   
1   2  College Park Towers         2        1.0   890                803   
2   3  College Park Towers         2        1.0   848                817   
3   4  College Park Towers         2        1.0   890                819   
4   5  College Park Towers         2        1.0   880                837   
5   6  College Park Towers         2        1.0   847                873   
6   7  College Park Towers         2        1.0   982                876   
7   8  College Park Towers         2        1.0   960                895   
8   9  College Park Towers         2        1.0   910                899   
9  10  College Park Towers         2        1.0   972                899   

   distance_to_campus_miles  walk_time_minutes  lease_type    availability  \
0                     

In [10]:
from google.colab import files
files.download('umd_housing_1000rows_synthetic.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>